Load Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.listdir("/content/drive/MyDrive/Shared Task/MultiClinSum/MultiClinSumDataset")


['multiclinsum_gs_train_en',
 'multiclinsum_gs_train_es',
 'multiclinsum_large-scale_train_en',
 'multiclinsum_large-scale_train_es',
 'multiclinsum_test_en',
 'multiclinsum_test_es']

In [ ]:
import os

path = r"/content/drive/MyDrive/Shared Task/MultiClinSum/MultiClinSumDataset/multiclinsum_gs_train_en/multiclinsum_gs_train_en"
os.listdir(path)


['summaries', 'fulltext']

In [ ]:
import os
import pandas as pd

base_path = "/content/drive/MyDrive/Shared Task/MultiClinSum/MultiClinSumDataset/multiclinsum_gs_train_en/multiclinsum_gs_train_en"
fulltext_path = os.path.join(base_path, "fulltext")
summary_path = os.path.join(base_path, "summaries")

data = []

# Extract numeric index from filename for sorting
def extract_index(fname):
    return int(fname.split("_")[-1].replace(".txt", ""))

# Traverse files in fulltext folder, sorted by index
for fname in sorted(os.listdir(fulltext_path), key=extract_index):
    if fname.endswith(".txt"):
        idx = fname.split("_")[-1].replace(".txt", "")
        fulltext_file = os.path.join(fulltext_path, fname)
        summary_file = os.path.join(summary_path, f"multiclinsum_gs_en_{idx}_sum.txt")

        try:
            with open(fulltext_file, "r", encoding="utf-8") as f:
                full_text = f.read().strip()
            with open(summary_file, "r", encoding="utf-8") as f:
                summary = f.read().strip()

            data.append({
                "id": int(idx),
                "full_text": full_text,
                "summary": summary
            })
        except Exception as e:
            print(f"=Error with index {idx}: {e}")

# Convert to DataFrame
df = pd.DataFrame(data)

# Sort by id
df = df.sort_values("id").reset_index(drop=True)

# Display info
print(f"=Loaded {len(df)} examples with columns: {list(df.columns)}")
df.head()


=Loaded 592 examples with columns: ['id', 'full_text', 'summary']


,id,full_text,summary
0,1,A 52-year-old female patient from Wolkiet (Nor...,A 52-year-old known female patient with a toxi...
1,2,"35-year-old man, occupation: sandblaster for e...","We present the case of a 35-year-old man, a sa..."
2,3,We present a case of a 59-year-old lady with a...,A 59-year-old woman was referred to the ophtha...
3,4,A 67-year-old man presented to our center with...,We present the case of a 67-year-old man who p...
4,5,A 64-year-old female patient presented to the ...,A 64-year-old woman with a confirmed diagnosis...


In [ ]:
print(" Total number of rows (samples):", len(df))
print(" Column names:", df.columns.tolist())
df.head(2)  # Optional: show first two rows

 Total number of rows (samples): 592
 Column names: ['id', 'full_text', 'summary']


,id,full_text,summary
0,1,A 52-year-old female patient from Wolkiet (Nor...,A 52-year-old known female patient with a toxi...
1,2,"35-year-old man, occupation: sandblaster for e...","We present the case of a 35-year-old man, a sa..."


In [ ]:
df.to_csv("/content/drive/MyDrive/Shared Task/MultiClinSum/Preprocessed/multiclinsum_gs_train_en.csv", index=False, encoding="utf-8")
print("save the file successfully：multiclinsum_gs_train_en.csv")

In [ ]:
# Install the OpenAI Python SDK if not already installed
!pip install openai --upgrade


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 720.5/720.5 kB 11.3 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.81.0
    Uninstalling openai-1.81.0:
      Successfully uninstalled openai-1.81.0


In [ ]:
import pandas as pd

# Read the previously saved training data from Google Drive
csv_path = "/content/drive/MyDrive/Shared Task/MultiClinSum/Preprocessed/multiclinsum_gs_train_en.csv"
df = pd.read_csv(csv_path)

# Preview the first few rows
df.head()


,id,full_text,summary
0,1,A 52-year-old female patient from Wolkiet (Nor...,A 52-year-old known female patient with a toxi...
1,2,"35-year-old man, occupation: sandblaster for e...","We present the case of a 35-year-old man, a sa..."
2,3,We present a case of a 59-year-old lady with a...,A 59-year-old woman was referred to the ophtha...
3,4,A 67-year-old man presented to our center with...,We present the case of a 67-year-old man who p...
4,5,A 64-year-old female patient presented to the ...,A 64-year-old woman with a confirmed diagnosis...


In [ ]:
# Select top 3 samples as few-shot examples (you can modify to use top 5)
few_shot_df = df.head(3)

# Build formatted text examples from full_text and summary
examples = ""
for i, row in few_shot_df.iterrows():
    examples += f"### Clinical Case Full Text {i+1}:\n{row['full_text']}\n\n"
    examples += f"### Corresponding Summary {i+1}:\n{row['summary']}\n\n"
    examples += "-" * 80 + "\n\n"


# Build the full meta prompt
meta_prompt = f"""
You are a medical AI assistant. Please help me to read a clinical case report. Your goal is to generate a **structured summary** that is **clinically sound**, **factually accurate**, and **concise**. The summary should cover the following five core components:


1. Patient Presentation: age, sex, relevant history.
2. Clinical Presentation: key symptoms and signs.
3. Diagnosis: relevant investigations, tests, conclusions.
4. Treatment/Intervention: medications, surgeries, therapies.
5. Outcome and Follow-up: results of treatment, current status.

Please note that the summary should be **significantly shorter** than the full text.

Below are several examples including full text and their corresponding summaries for your reference:

Before drafting your prompt, please think aloud:
- What common structure or patterns do you observe in the examples?
- What information is emphasized?
- How can a language model be guided to produce similar quality outputs?
- What errors should be avoided?

You should also consider how your prompt may influence **ROUGE-L** (for overlap) and **BERTScore** (for semantic fidelity).

Once you've analyzed the examples, propose an **initial version of the prompt**. I will then test it on new examples, return the model outputs and ground-truth references, and invite you to revise the prompt for better performance.


{examples}


"""
print(meta_prompt[:1500])  # Truncate output to avoid flooding the screen



You are a medical AI assistant. Please help me to read a clinical case report. Your goal is to generate a **structured summary** that is **clinically sound**, **factually accurate**, and **concise**. The summary should cover the following five core components:


1. Patient Presentation: age, sex, relevant history.
2. Clinical Presentation: key symptoms and signs.
3. Diagnosis: relevant investigations, tests, conclusions.
4. Treatment/Intervention: medications, surgeries, therapies.
5. Outcome and Follow-up: results of treatment, current status.

Please note that the summary should be **significantly shorter** than the full text. 

Below are several examples including full text and their corresponding summaries for your reference:

Before drafting your prompt, please think aloud:
- What common structure or patterns do you observe in the examples?
- What information is emphasized?
- How can a language model be guided to produce similar quality outputs?
- What errors should be avoided?



In [ ]:
import os
import pandas as pd

# set up the dictionary
output_dir = "/content/drive/MyDrive/Shared Task/MultiClinSum/Preprocessed"
output_file = os.path.join(output_dir, "meta_prompt.txt")
os.makedirs(output_dir, exist_ok=True)

with open(output_file, "w", encoding="utf-8") as f:
    f.write(meta_prompt)

print("Meta prompt saved to:", output_file)

Meta prompt saved to: /content/drive/MyDrive/Shared Task/MultiClinSum/Preprocessed/meta_prompt.txt


In [ ]:
!pip install --upgrade openai

from openai import OpenAI

client = OpenAI(api_key="sk-")

In [ ]:
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are an expert medical NLP researcher."},
        {"role": "user", "content": meta_prompt}
    ],
    temperature=0.7,
    max_tokens=1500
)

# extract the generated prompt
generated_prompt = response.choices[0].message.content
print("-----Generated Prompt-----\n")
print(generated_prompt)

-----Generated Prompt-----

**Think Aloud:**

1. **Common Structure and Patterns:**
   - Each summary follows a structured format corresponding to the five key components: patient presentation, clinical presentation, diagnosis, treatment/intervention, and outcome.
   - The summaries start by introducing the patient's age, sex, and relevant medical history.
   - Clinical symptoms and findings are briefly described, focusing on the most pertinent details.
   - Diagnostic procedures and their results are succinctly mentioned, emphasizing key findings that led to the diagnosis.
   - Treatment details are provided, highlighting major interventions and medications.
   - Outcomes and follow-up information are clearly stated, summarizing the patient's response to treatment and current status.

2. **Information Emphasis:**
   - The emphasis is placed on critical details that influence diagnosis and treatment.
   - Diagnostic findings are succinct but precise, ensuring understanding of the condi

In [ ]:
import os

output_dir = "/content/drive/MyDrive/Shared Task/MultiClinSum/Preprocessed"
os.makedirs(output_dir, exist_ok=True)

# Auto-generate file name
existing_files = [f for f in os.listdir(output_dir) if f.startswith("prompt_v") and f.endswith(".txt")]
file_index = len(existing_files) + 1
file_name = f"prompt_v{file_index}.txt"
file_path = os.path.join(output_dir, file_name)

# Write the generated prompt from GPT
with open(file_path, "w", encoding="utf-8") as f:
    f.write(generated_prompt)

print(f"Prompt saved to: {file_path}")


Prompt saved to: /content/drive/MyDrive/Shared Task/MultiClinSum/Preprocessed/prompt_v1.txt


# Phase 2. Small-scale Inference + Self-Reflection

In [ ]:
!pip install bert-score rouge-score


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 110.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 68.1 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=2

In [ ]:
import pandas as pd
from openai import OpenAI
import os
from tqdm import tqdm
from bert_score import score as bertscore
from rouge_score import rouge_scorer

client = OpenAI(api_key="sk-")

data_path = "/content/drive/MyDrive/Shared Task/MultiClinSum/Preprocessed/multiclinsum_gs_train_en.csv"
prompt_path = "/content/drive/MyDrive/Shared Task/MultiClinSum/Preprocessed/prompt_v1.txt"

output_base = "/content/drive/MyDrive/Shared Task/MultiClinSum/Phase2_Reflection/epoch2.0"
summary_output_dir = os.path.join(output_base, "Generated_Summary_v2.0")
os.makedirs(summary_output_dir, exist_ok=True)


## 1. Read the data and prepare the samples (excluding few-shot examples), and load the meta prompt.

In [ ]:
# Load the first 50 full texts and gold summaries
df = pd.read_csv(data_path)

# Select entries 4 to 53 (skip the first 3 few-shot examples)
sample_df = df.iloc[3:53].copy().reset_index(drop=True)

# Step 3: Load the initial meta prompt (including few-shot examples)
with open(prompt_path, 'r', encoding='utf-8') as f:
    meta_prompt = f.read()


## 2: GPT Function

In [ ]:
# generate the summary using GPT-4o
def generate_summary(full_text, meta_prompt):
    prompt_input = meta_prompt + f"\n\nText 4:\n{full_text}\n\nSummary 4:"
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt_input}],
        temperature=0.5,
        max_tokens=2500
    )
    return response.choices[0].message.content.strip()


# reflection
def reflect_and_suggest_prompt(generated, gold, original_prompt):
    prompt = f"""
You are a prompt engineer and medical NLP reviewer.

Here is a generated summary:
---\n{generated}\n---

Here is the gold standard summary:
---\n{gold}\n---

Here is the original prompt (few-shot included, truncated):
---\n{original_prompt[:800]}\n---

Please complete the following 3 steps:
1. Identify missing or inaccurate information in the generated summary compared to the gold summary.
2. Briefly describe any issues in structure, clarity, or style.
3. Suggest 2–3 concrete changes that can be made to the prompt to improve future outputs.
4. Finally, write a revised version or snippet of the prompt that incorporates those changes.

Keep your feedback concise and structured.
"""

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5,
        max_tokens=2500
    )
    return response.choices[0].message.content.strip()


## 3: Main Loop

In [ ]:
generated_summaries = []
reflection_notes = []
prompt_suggestions = []

for idx, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    full_text = row['full_text']
    gold_summary = row['summary']

    try:
        # 1. Generate summary
        generated = generate_summary(full_text, meta_prompt)

        # 2. Automatic reflection (keep full content without keyword splitting)
        reflection_result = reflect_and_suggest_prompt(generated, gold_summary, meta_prompt)

        if "Suggested Prompt Update" in reflection_result:
            before, after = reflection_result.split("Suggested Prompt Update", 1)
            reflection = before.strip()
            suggestion = after.strip().replace("```", "").replace("markdown", "").replace("Prompt Update", "")
        else:
            reflection = reflection_result
            suggestion = ""

        generated_summaries.append(generated)
        reflection_notes.append(reflection)
        prompt_suggestions.append(suggestion)

        # 3. Save the generated summary as a .txt file
        file_name = f"multiclinsum_gs_train_en_v1_{idx+3}.txt"
        with open(os.path.join(summary_output_dir, file_name), 'w', encoding='utf-8') as f:
            f.write(generated)

    except Exception as e:
        print(f"Error at index {idx}: {e}")
        generated_summaries.append("ERROR")
        reflection_notes.append("ERROR")
        prompt_suggestions.append("ERROR")

100%|██████████| 50/50 [11:41<00:00, 14.02s/it]


## 4: save the result

In [ ]:
sample_df['generated_summary'] = generated_summaries
sample_df['reflection'] = reflection_notes
sample_df['suggested_prompt_update'] = prompt_suggestions

intermediate_csv = os.path.join(output_base, "phase2_reflection2_with_prompt_v2.0.csv")
sample_df.to_csv(intermediate_csv, index=False)

## 5: Evaluation（BERTScore + ROUGE-L-Sum）

In [ ]:
import os
import pandas as pd
from bert_score import score as bertscore
from rouge_score import rouge_scorer
import torch
from tqdm import tqdm

# 0: Set paths
eval_file_path = "/content/drive/MyDrive/Shared Task/MultiClinSum/Phase2_Reflection/epoch2.0/phase2_reflection2_with_prompt_v2.0.csv"
output_dir = "/content/drive/MyDrive/Shared Task/MultiClinSum/Phase2_Reflection/epoch2.0/Phase2_Reflection2_Eval2.0"
os.makedirs(output_dir, exist_ok=True)

log_file_path = os.path.join(output_dir, "evaluation_log.txt")
log_lines = []

# 1: Load data
dfeval = pd.read_csv(eval_file_path)

# 2: Mark invalid summaries (empty or fallback)
def get_exclusion_reason(text):
    text = str(text).strip().lower()
    if text == "":
        return "empty"
    if "not generated" in text or "failed" in text:
        return "fallback summary"
    return None

dfeval['reason_for_exclusion'] = dfeval['generated_summary'].apply(get_exclusion_reason)
dfeval['valid_for_eval'] = dfeval['reason_for_exclusion'].isnull()

# 3: Split valid and excluded samples
eval_df = dfeval[dfeval['valid_for_eval']].copy()
excluded_df = dfeval[~dfeval['valid_for_eval']].copy()

# 4: Output evaluation statistics
line1 = f"\n Evaluation Summary:"
line2 = f" Total samples: {len(dfeval)}"
line3 = f" Valid for evaluation: {len(eval_df)}"
line4 = f" Excluded from evaluation: {len(excluded_df)}"
print(line1); print(line2); print(line3); print(line4)
log_lines.extend([line1, line2, line3, line4])

if len(excluded_df) > 0:
    excluded_idx_str = f" Excluded sample indices (in current df): {excluded_df.index.tolist()}"
    print(excluded_idx_str)
    log_lines.append(excluded_idx_str)

    if 'original_index' in excluded_df.columns:
        original_idx_str = f" Corresponding original indices: {excluded_df['original_index'].tolist()}"
        print(original_idx_str)
        log_lines.append(original_idx_str)

    reason_str = " Reason: Summary is empty or marked fallback."
    print(reason_str)
    log_lines.append(reason_str)

# 5: BERTScore Evaluation
P, R, F1 = bertscore(
    cands=eval_df['generated_summary'].tolist(),
    refs=eval_df['summary'].tolist(),
    lang="en",
    model_type="roberta-large",
    device="cuda" if torch.cuda.is_available() else "cpu"
)
eval_df['bertscore_f1'] = F1

# 6: ROUGE-L-Sum Evaluation
scorer = rouge_scorer.RougeScorer(['rougeLsum'], use_stemmer=True)
rouge_scores = [
    scorer.score(ref, pred)['rougeLsum'].fmeasure
    for ref, pred in tqdm(zip(eval_df['summary'], eval_df['generated_summary']), total=len(eval_df), desc="Calculating ROUGE-L-Sum")
]
eval_df['rougeLsum_f1'] = rouge_scores

# 7: Output average scores
bert_line = f"\n Avg BERTScore F1: {F1.mean():.4f}"
rouge_line = f" Avg ROUGE-L-Sum F1: {sum(rouge_scores)/len(rouge_scores):.4f}"
print(bert_line)
print(rouge_line)
log_lines.extend([bert_line, rouge_line])

# 8: Save evaluation table
eval_df.to_csv(os.path.join(output_dir, "evaluated_samples2.0.csv"), index=False)
excluded_df.to_csv(os.path.join(output_dir, "excluded_samples2.0.csv"), index=False)
dfeval.to_csv(os.path.join(output_dir, "full_with_eval_flags2.0.csv"), index=False)

# 9: Save log file
with open(log_file_path, "w", encoding="utf-8") as f:
    f.write("\n".join(log_lines))

print(f"\n All evaluation information were saved in: {log_file_path}")



 Evaluation Summary:
 Total samples: 50
 Valid for evaluation: 49
 Excluded from evaluation: 1
 Excluded sample indices (in current df): [2]
 Reason: Summary is empty or marked fallback.


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Calculating ROUGE-L-Sum: 100%|██████████| 49/49 [00:00<00:00, 102.42it/s]



 Avg BERTScore F1: 0.8592
 Avg ROUGE-L-Sum F1: 0.3046

 All evaluation information were saved in: /content/drive/MyDrive/Shared Task/MultiClinSum/Phase2_Reflection/epoch2.0/Phase2_Reflection2_Eval2.0/evaluation_log.txt


## 6. Update the Prompt

In [ ]:
!pip install --upgrade openai


import pandas as pd
from pathlib import Path
from openai import OpenAI
import os

In [ ]:
client = OpenAI(api_key="sk-")

batch_size = 5
num_rounds = 3
output_dir = Path("/content/drive/MyDrive/Shared Task/MultiClinSum/Phase2_Reflection/epoch2.0/Phase2_Reflection2_Eval2.0")
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Upload the data
eval_path = "/content/drive/MyDrive/Shared Task/MultiClinSum/Phase2_Reflection/epoch2.0/Phase2_Reflection2_Eval2.0/evaluated_samples2.0.csv"
extra_path = "/content/drive/MyDrive/Shared Task/MultiClinSum/Phase2_Reflection/epoch2.0/phase2_reflection2_with_prompt_v2.0.csv"
prompt_path = "/content/drive/MyDrive/Shared Task/MultiClinSum/Preprocessed/prompt_v1.txt"

df_eval = pd.read_csv(eval_path)
df_extra = pd.read_csv(extra_path)
original_prompt = Path(prompt_path).read_text()

# Extract all reflections (including reflections with low ROUGEScore and invalid index)
lowest_reflections = df_eval.sort_values(by="rougeLsum_f1").head(batch_size * num_rounds)["reflection"].tolist()
extra_reflection = df_extra.iloc[31]["reflection"]
all_reflections = lowest_reflections + [extra_reflection]

In [ ]:
# generate new prompt using meta prompt
def build_meta_prompt(current_prompt, reflection_batch):
    combined = "\n\n---\n\n".join(reflection_batch)
    meta_prompt = f"""
You are a medical AI assistant. Your task is to generate a **structured summary** from a clinical case report that is **clinically sound**, **factually accurate**, and **concise**. Each summary must follow this structure:

1. Patient Presentation: age, sex, relevant history
2. Clinical Presentation: key symptoms and signs
3. Diagnosis: relevant investigations, tests, conclusions
4. Treatment/Intervention: medications, surgeries, therapies
5. Outcome and Follow-up: results of treatment, current status

The summary should be significantly shorter than the full text and maintain fidelity to the original case.

This prompt design process follows a **reflection-based iteration framework**. Initially, a base prompt (**Prompt v1**) was used to guide generation. We evaluated the generated summaries and collected human reflections, especially focusing on those with low ROUGE-L-Sum scores.

**Current priority: improving ROUGE-L-Sum.**
The summaries generated using Prompt v1 often had low ROUGE-L-Sum scores, indicating poor alignment with gold summaries in terms of phrasing, wording, and structure.

Now, your goal is to revise **Prompt v1**, based on the batch of reflections below.
**Do not make unnecessary edits to Prompt v1. Only revise parts specifically suggested by reflection feedback.**

--- Current Prompt ---
{current_prompt}
-----------------------

--- Reflections (current batch) ---
{combined}
-----------------------

Please carefully revise the prompt to:

- Improve ROUGE-L-Sum (expression/structure overlap with gold)
- Preserve factual accuracy and clarity
- Maintain all 5 required sections
- Modify ONLY where reflection suggests a weakness

Output ONLY the revised prompt (no explanations or other text).
"""
    return meta_prompt


In [ ]:
# iteratively executes
current_prompt = original_prompt

for i in range(num_rounds):
    print(f"\n Starting Iteration {i+1}/{num_rounds}...")

    start_idx = i * batch_size
    end_idx = start_idx + batch_size
    batch_reflections = all_reflections[start_idx:end_idx]

    meta_prompt = build_meta_prompt(current_prompt, batch_reflections)

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a helpful medical NLP assistant."},
            {"role": "user", "content": meta_prompt}
        ],
        temperature=0.5,
        max_tokens=1500
    )

    revised_prompt = response.choices[0].message.content

    print(f" Revised prompt length: {len(revised_prompt)} characters")
    print(f" Token usage: {response.usage.total_tokens} tokens")

    # save meta prompt + revised prompt
    output_path = output_dir / f"{i+1:02d}_prompt_v2_iter.txt"
    meta_path = output_dir / f"{i+1:02d}_meta_prompt_used.txt"
    output_path.write_text(revised_prompt)
    meta_path.write_text(meta_prompt)

    print(f" Saved revised prompt to: {output_path.name}")

    # Updated for the next round
    current_prompt = revised_prompt


 Starting Iteration 1/3...
 Revised prompt length: 1241 characters
 Token usage: 3686 tokens
 Saved revised prompt to: 01_prompt_v2_iter.txt

 Starting Iteration 2/3...
 Revised prompt length: 1914 characters
 Token usage: 3489 tokens
 Saved revised prompt to: 02_prompt_v2_iter.txt

 Starting Iteration 3/3...
 Revised prompt length: 1624 characters
 Token usage: 3474 tokens
 Saved revised prompt to: 03_prompt_v2_iter.txt
